In [127]:
import dask.dataframe as dd
import pandas as pd
import pandas_ta as ta
import numpy as np
import os
import glob
import sys
from pathlib import Path
import pyarrow as pa


In [128]:
sector_file = 'stock_data/meta/ind_nifty500list_filtered_final.csv'
stock_sector = pd.read_csv(sector_file, dtype={'Symbol': str})
print(f"Loaded {len(stock_sector)} sector mappings")

# 2. Load Raw Stock Data
files = glob.glob('stock_data/raw/*.csv')
print(f"Found {len(files)} stock files")

dfs = []
for f in files:
    try:
        ticker = Path(f).stem
        if ticker not in stock_sector['Symbol'].values:
            print(f"Ticker {ticker} not found in sector mapping, skipping")
            continue

        # Read with date parsing and validation
        df = pd.read_csv(
            f,
            usecols=['Date', 'Close', 'Stock Splits'],
            parse_dates=['Date'],
            index_col='Date',
            dayfirst=True  # Important for non-US date formats
        )
        
        # Handle splits and calculate adjusted close
        df['Split Factor'] = (1 + df['Stock Splits']).replace(0, 1).cumprod()
        df['Adj Close'] = df['Close'] / df['Split Factor']
        
        dfs.append(df[['Adj Close']].rename(columns={'Adj Close': ticker}))
        
    except Exception as e:
        print(f"Error processing {ticker}: {str(e)}")

# 3. Create Consolidated DataFrame
if not dfs:
    raise ValueError("No valid stock data processed")

master_df = pd.concat(dfs, axis=1)
print(f"Master DataFrame shape: {master_df.shape}")

# 4. Merge with Sector Information (preserve date index)
merged_df = (
    master_df.T
    .reset_index()
    .rename(columns={'index': 'Symbol'})
    .merge(stock_sector[['Symbol', 'Industry']],  # Verify actual column name
           on='Symbol',
           how='left')
    .set_index(['Industry', 'Symbol'])
    .T
    .sort_index(axis=1)
)

# Convert index to proper datetime
merged_df.index = pd.to_datetime(merged_df.index)
print("\nFinal Merged DataFrame:")
print(f"Index: {merged_df.index.name}")
print(f"Columns: {merged_df.columns.names}")
print(f"Shape: {merged_df.shape}")
print(merged_df.head(2))


Loaded 445 sector mappings
Found 505 stock files
Ticker CELLO not found in sector mapping, skipping
Ticker HONASA not found in sector mapping, skipping
Ticker TBOTEK not found in sector mapping, skipping
Ticker MANYAVAR not found in sector mapping, skipping
Ticker RRKABEL not found in sector mapping, skipping
Ticker IREDA not found in sector mapping, skipping
Ticker ADANIENSOL not found in sector mapping, skipping
Ticker SYRMA not found in sector mapping, skipping
Ticker SIGNATURE not found in sector mapping, skipping
Ticker INOXINDIA not found in sector mapping, skipping
Ticker RAINBOW not found in sector mapping, skipping
Ticker DELHIVERY not found in sector mapping, skipping
Ticker FIVESTAR not found in sector mapping, skipping
Ticker LLOYDSME not found in sector mapping, skipping
Ticker JnKBANK not found in sector mapping, skipping
Ticker INDGN not found in sector mapping, skipping
Ticker NSLNISP not found in sector mapping, skipping
Ticker TATATECH not found in sector mapping, ski

In [129]:
merged_df.head()

Industry                  Automobile and Auto Components              \
Symbol                                        APOLLOTYRE  ASAHIINDIA   
2022-01-10 00:00:00+05:30                     225.580246  547.070801   
2022-01-11 00:00:00+05:30                     224.949585  566.683289   
2022-01-12 00:00:00+05:30                     226.696014  559.700012   
2022-01-13 00:00:00+05:30                     227.957336  550.983398   
2022-01-14 00:00:00+05:30                     224.707031  570.843506   

Industry                                                                       \
Symbol                      BAJAJ-AUTO   BALKRISIND  BHARATFORG      BOSCHLTD   
2022-01-10 00:00:00+05:30  3186.578613  2343.807129  748.324463  16422.919922   
2022-01-11 00:00:00+05:30  3188.109131  2358.665039  750.469543  16297.809570   
2022-01-12 00:00:00+05:30  3191.216797  2387.699219  741.499329  16409.439453   
2022-01-13 00:00:00+05:30  3201.328125  2405.821045  748.812012  16401.472656   
2022-01-14 00:00:00+05:30  3184.445068  2425.745361  761.584717  16485.757812   

Industry                                                                      \
Symbol                         CEATLTD    CIEINDIA    CRAFTSMAN    EICHERMOT   
2022-01-10 00:00:00+05:30  1125.336304  221.650314  2311.613037  2736.037354   
2022-01-11 00:00:00+05:30  1126.414185  224.717316  2302.476562  2715.484619   
2022-01-12 00:00:00+05:30  1107.895386  219.849091  2310.520752  2736.425781   
2022-01-13 00:00:00+05:30  1116.664917  220.579315  2328.644775  2752.751709   
2022-01-14 00:00:00+05:30  1116.615723  221.455612  2324.126221  2740.118896   

Industry                   ... Telecommunication                           \
Symbol                     ...             ROUTE     TATACOMM    TEJASNET   
2022-01-10 00:00:00+05:30  ...       1767.905029  1421.090332  446.899994   
2022-01-11 00:00:00+05:30  ...       1817.629517  1415.016235  469.200012   
2022-01-12 00:00:00+05:30  ...       1830.158325  1490.009155  491.000000   
2022-01-13 00:00:00+05:30  ...       1862.771606  1478.339355  515.549988   
2022-01-14 00:00:00+05:30  ...       1843.612915  1478.100342  509.149994   

Industry                                Textiles                            \
Symbol                           TTML   ALOKINDS     KPRMILL       PAGEIND   
2022-01-10 00:00:00+05:30  276.350006  29.799999  717.887878  42093.679688   
2022-01-11 00:00:00+05:30  290.149994  30.000000  721.382324  42095.378906   
2022-01-12 00:00:00+05:30  275.649994  29.000000  733.932800  42838.976562   
2022-01-13 00:00:00+05:30  261.899994  28.750000  728.125183  42822.011719   
2022-01-14 00:00:00+05:30  248.850006  30.049999  744.514648  43031.667969   

Industry                                                      
Symbol                       TRIDENT         VTL  WELSPUNLIV  
2022-01-10 00:00:00+05:30  51.811359  515.310852  152.255219  
2022-01-11 00:00:00+05:30  54.382523  507.546326  148.519806  
2022-01-12 00:00:00+05:30  57.099220  510.023102  149.615509  
2022-01-13 00:00:00+05:30  59.912949  516.804810  153.251328  
2022-01-14 00:00:00+05:30  62.872211  528.766113  150.611618  

[5 rows x 445 columns]

In [130]:
def calculate_sma(data, window):
    return data.rolling(window=window).mean()

def calculate_ema(data, window):
    return data.ewm(span=window, adjust=False).mean()


In [131]:
def calculate_bollinger_bands(data, window, num_std_dev):
    sma = calculate_sma(data, window)
    rolling_std = data.rolling(window=window).std()
    upper_band = sma + (rolling_std * num_std_dev)
    lower_band = sma - (rolling_std * num_std_dev)
    return upper_band, lower_band

In [ ]:
def calculate_advanced_momentum_indicator(price, window=20, volatility_factor=2, smoothing=5):

    reference = price.rolling(window=window).mean()
    deviation = price.rolling(window=window).std() * volatility_factor
    
    upper_reference = reference + deviation
    lower_reference = reference - deviation
    
    raw_position = (price - lower_reference) / (upper_reference - lower_reference).replace(0, np.finfo(float).eps)
    
    trend_component = price.diff(3).ewm(span=smoothing).mean()

    sequence = np.arange(1, 3)[::-1] 
    weights = sequence / sequence.sum()

    adaptive_trend = pd.Series(0, index=price.index)
    for i, w in enumerate(weights):
        if i > 0:
            offset_value = price.shift(-(i+1))
            last_valid = price.iloc[-1] if len(price) > 0 else 0
            offset_value = offset_value.fillna(last_valid)
        else:
            offset_value = price

        adaptive_trend += w * (offset_value - reference) / deviation.replace(0, np.finfo(float).eps)

    final_signal = (raw_position * 0.1) + (adaptive_trend * 0.9)
    normalized_momentum = final_signal.clip(-2, 2) / 2
    
    return normalized_momentum

In [133]:
def calculate_rsi(data, window):
    delta = data.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.rolling(window=window).mean()
    avg_loss = loss.rolling(window=window).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

In [134]:
def calculate_macd(data, fast_window, slow_window, signal_window):
    ema_fast = calculate_ema(data, fast_window)
    ema_slow = calculate_ema(data, slow_window)
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal_window, adjust=False).mean()
    macd_histogram = macd_line - signal_line
    return macd_line, signal_line, macd_histogram

In [135]:
def approximate_atr(data, window):
    return data.diff().abs().rolling(window=window).mean()

In [136]:
merged_df.head()

Industry                  Automobile and Auto Components              \
Symbol                                        APOLLOTYRE  ASAHIINDIA   
2022-01-10 00:00:00+05:30                     225.580246  547.070801   
2022-01-11 00:00:00+05:30                     224.949585  566.683289   
2022-01-12 00:00:00+05:30                     226.696014  559.700012   
2022-01-13 00:00:00+05:30                     227.957336  550.983398   
2022-01-14 00:00:00+05:30                     224.707031  570.843506   

Industry                                                                       \
Symbol                      BAJAJ-AUTO   BALKRISIND  BHARATFORG      BOSCHLTD   
2022-01-10 00:00:00+05:30  3186.578613  2343.807129  748.324463  16422.919922   
2022-01-11 00:00:00+05:30  3188.109131  2358.665039  750.469543  16297.809570   
2022-01-12 00:00:00+05:30  3191.216797  2387.699219  741.499329  16409.439453   
2022-01-13 00:00:00+05:30  3201.328125  2405.821045  748.812012  16401.472656   
2022-01-14 00:00:00+05:30  3184.445068  2425.745361  761.584717  16485.757812   

Industry                                                                      \
Symbol                         CEATLTD    CIEINDIA    CRAFTSMAN    EICHERMOT   
2022-01-10 00:00:00+05:30  1125.336304  221.650314  2311.613037  2736.037354   
2022-01-11 00:00:00+05:30  1126.414185  224.717316  2302.476562  2715.484619   
2022-01-12 00:00:00+05:30  1107.895386  219.849091  2310.520752  2736.425781   
2022-01-13 00:00:00+05:30  1116.664917  220.579315  2328.644775  2752.751709   
2022-01-14 00:00:00+05:30  1116.615723  221.455612  2324.126221  2740.118896   

Industry                   ... Telecommunication                           \
Symbol                     ...             ROUTE     TATACOMM    TEJASNET   
2022-01-10 00:00:00+05:30  ...       1767.905029  1421.090332  446.899994   
2022-01-11 00:00:00+05:30  ...       1817.629517  1415.016235  469.200012   
2022-01-12 00:00:00+05:30  ...       1830.158325  1490.009155  491.000000   
2022-01-13 00:00:00+05:30  ...       1862.771606  1478.339355  515.549988   
2022-01-14 00:00:00+05:30  ...       1843.612915  1478.100342  509.149994   

Industry                                Textiles                            \
Symbol                           TTML   ALOKINDS     KPRMILL       PAGEIND   
2022-01-10 00:00:00+05:30  276.350006  29.799999  717.887878  42093.679688   
2022-01-11 00:00:00+05:30  290.149994  30.000000  721.382324  42095.378906   
2022-01-12 00:00:00+05:30  275.649994  29.000000  733.932800  42838.976562   
2022-01-13 00:00:00+05:30  261.899994  28.750000  728.125183  42822.011719   
2022-01-14 00:00:00+05:30  248.850006  30.049999  744.514648  43031.667969   

Industry                                                      
Symbol                       TRIDENT         VTL  WELSPUNLIV  
2022-01-10 00:00:00+05:30  51.811359  515.310852  152.255219  
2022-01-11 00:00:00+05:30  54.382523  507.546326  148.519806  
2022-01-12 00:00:00+05:30  57.099220  510.023102  149.615509  
2022-01-13 00:00:00+05:30  59.912949  516.804810  153.251328  
2022-01-14 00:00:00+05:30  62.872211  528.766113  150.611618  

[5 rows x 445 columns]

In [ ]:
def add_enhanced_features(df):
    """Add features focused on short-term (daily) return prediction"""
    print(f"DataFrame structure: {df.columns.nlevels} levels")
    print(f"Column levels: {[name for name in df.columns.names]}")

    # Store new columns as tuples first
    new_columns = {}

    # Define feature names list used in exception handling
    feature_names = ['price', 'return_1d', 'return_3d', 'return_5d', 
                     'SMA_5', 'SMA_10', 'SMA_20', 'EMA_5', 'EMA_10', 'EMA_20',
                     'WMA_5', 'WMA_10', 'WMA_20', 'dist_from_sma10', 'dist_from_ema10',
                     'dist_from_sma20', 'MACD_hist','adv_momentum', 'RSI_14',
                     'volatility_10d', 'volatility_20d', 'BB_width', 'BB_pos',
                     'ema5_ema10_cross', 'ema10_ema20_cross', 'return_ratio']


    # Process each industry and symbol
    print("Starting initial feature calculation for all stocks...")
    for industry in df.columns.levels[0]:  # Iterate over industries
        symbols = df[industry].columns  # Get symbols for this industry

        for symbol in symbols:  # Iterate over symbols within each industry
            try:
                # Get base price data - check for NaN/empty series early
                price = df[(industry, symbol)]
                if price.isnull().all() or price.empty:
                    print(f"Skipping {industry} - {symbol}: Price data is all NaN or empty.")
                    # Add NaN placeholders explicitly and continue
                    for feat in feature_names:
                        if (industry, symbol, feat) not in new_columns:
                            new_columns[(industry, symbol, feat)] = pd.Series(np.nan, index=df.index)
                    continue # Skip to next symbol

                #--------------------------------------------------------------
                # 1. ESSENTIAL PRICE-BASED FEATURES (Short-Term Focus)
                #--------------------------------------------------------------
                new_columns[(industry, symbol, 'price')] = price
                return_1d = price.pct_change(1) # Calculate once
                new_columns[(industry, symbol, 'return_1d')] = return_1d
                new_columns[(industry, symbol, 'return_3d')] = price.pct_change(3)
                new_columns[(industry, symbol, 'return_5d')] = price.pct_change(5)
                

                #--------------------------------------------------------------
                # 2. KEY TECHNICAL INDICATORS (Short-Term Focus)
                #--------------------------------------------------------------
                sma5 = price.rolling(window=5).mean()
                sma10 = price.rolling(window=10).mean()
                sma20 = price.rolling(window=20).mean()
                new_columns[(industry, symbol, 'SMA_5')] = sma5
                new_columns[(industry, symbol, 'SMA_10')] = sma10
                new_columns[(industry, symbol, 'SMA_20')] = sma20

                ema5 = price.ewm(span=5, adjust=False).mean()
                ema10 = price.ewm(span=10, adjust=False).mean()
                ema20 = price.ewm(span=20, adjust=False).mean()
                new_columns[(industry, symbol, 'EMA_5')] = ema5
                new_columns[(industry, symbol, 'EMA_10')] = ema10
                new_columns[(industry, symbol, 'EMA_20')] = ema20

                for window in [5, 10, 20]:
                    weights = np.arange(1, window + 1)
                    # Ensure window size is not larger than available data points for rolling
                    if len(price) >= window:
                        wma = price.rolling(window=window).apply(
                            lambda x: np.sum(weights * x) / weights.sum(), raw=True
                        )
                    else:
                        wma = pd.Series(np.nan, index=price.index) # Assign NaNs if not enough data
                    new_columns[(industry, symbol, f'WMA_{window}')] = wma

                # Use calculated MAs, handle division by zero
                new_columns[(industry, symbol, 'dist_from_sma10')] = (price - sma10) / sma10.replace(0, np.finfo(float).eps)
                new_columns[(industry, symbol, 'dist_from_ema10')] = (price - ema10) / ema10.replace(0, np.finfo(float).eps)
                new_columns[(industry, symbol, 'dist_from_sma20')] = (price - sma20) / sma20.replace(0, np.finfo(float).eps)

                ema12 = price.ewm(span=12, adjust=False).mean()
                ema26 = price.ewm(span=26, adjust=False).mean()
                macd_line = ema12 - ema26
                signal_line = macd_line.ewm(span=9, adjust=False).mean()
                new_columns[(industry, symbol, 'MACD_hist')] = macd_line - signal_line

                delta = price.diff()


                gain = delta.where(delta > 0, 0).fillna(0)
                loss = -delta.where(delta < 0, 0).fillna(0)
                avg_gain = gain.ewm(alpha=1/14, adjust=False).mean()
                avg_loss = loss.ewm(alpha=1/14, adjust=False).mean()
                rs = avg_gain / avg_loss.replace(0, np.finfo(float).eps)
                new_columns[(industry, symbol, 'RSI_14')] = 100 - (100 / (1 + rs))

                new_columns[(industry, symbol, 'volatility_10d')] = price.pct_change().rolling(10).std()
                new_columns[(industry, symbol, 'volatility_20d')] = price.pct_change().rolling(20).std()

                std20 = price.rolling(window=20).std()
                upper_band = sma20 + (std20 * 2)
                lower_band = sma20 - (std20 * 2)
                bb_width_denom = sma20.replace(0, np.finfo(float).eps)
                bb_pos_denom = (upper_band - lower_band).replace(0, np.finfo(float).eps)

                new_columns[(industry, symbol, 'BB_width')] = (upper_band - lower_band) / bb_width_denom
                new_columns[(industry, symbol, 'BB_pos')] = (price - lower_band) / bb_pos_denom

                momentum_indicator = calculate_advanced_momentum_indicator(
                    price, 
                    window=15,  
                    volatility_factor=1.8,  
                    smoothing=4
                )
                new_columns[(industry, symbol, 'adv_momentum')] = momentum_indicator

                new_columns[(industry, symbol, 'ema5_ema10_cross')] = (ema5 > ema10).astype(int)
                new_columns[(industry, symbol, 'ema10_ema20_cross')] = (ema10 > ema20).astype(int)


                #--------------------------------------------------------------
                # 4. TARGET VARIABLE (Daily Return)
                #--------------------------------------------------------------
                # Assign the previously calculated return_1d
                new_columns[(industry, symbol, 'return_ratio')] = return_1d


            except Exception as e:
                print(f"Failed during feature calculation for {industry} - {symbol}: {e}")
                # Add NaN placeholders for ALL features for this symbol if *any* error occurs
                for feat in feature_names:
                    if (industry, symbol, feat) not in new_columns:
                        new_columns[(industry, symbol, feat)] = pd.Series(np.nan, index=df.index)
                # Continue to the next symbol, this symbol's data might be partial/NaN
                continue
    # --- End of Industry/Symbol Loops ---

    # --- Create result_df AFTER processing all symbols ---
    print("Constructing final DataFrame from calculated features...")
    result_df = pd.DataFrame(new_columns, index=df.index)
    # Ensure proper MultiIndex structure
    result_df.columns = pd.MultiIndex.from_tuples(result_df.columns, names=['Industry', 'Symbol', 'Feature'])
    # Make a copy to avoid fragmentation warnings when adding relative features later
    result_df = result_df.copy()
    print(f"Initial result_df shape: {result_df.shape}")



    print("Finished calculating features.")
    # Sort columns for consistency
    result_df = result_df.sort_index(axis=1)
    return result_df

# Apply the function to your DataFrame
enhanced_df = add_enhanced_features(merged_df)

# --- Post-processing ---
print("Starting post-processing (fillna, dropna)...")
# Fill NaNs carefully - forward fill is often appropriate for time series
enhanced_df = enhanced_df.ffill()
# Backward fill any remaining NaNs at the beginning
enhanced_df = enhanced_df.bfill()

# Drop rows where the target variable 'return_ratio' is NaN, as these cannot be used for training/evaluation
print(f"Shape before dropping rows with NaN target: {enhanced_df.shape}")
enhanced_df = enhanced_df.dropna(subset=pd.MultiIndex.from_tuples(
    [(ind, sym, 'return_ratio') for ind, sym in enhanced_df.columns.droplevel(2).unique()]
), how='all') # Drop row if ALL return_ratio are NaN (less aggressive)
# Or drop if ANY return_ratio is NaN for a more strict approach:
# enhanced_df = enhanced_df.dropna(subset=[col for col in enhanced_df.columns if col[2] == 'return_ratio'], how='any')

# Optionally, drop columns (features) that are entirely NaN across all stocks/dates
# enhanced_df = enhanced_df.dropna(axis=1, how='all')

# Drop rows that *still* have NaNs in *any* feature column (most strict)
# Consider if this is too aggressive - maybe some features can be NaN?
# enhanced_df = enhanced_df.dropna(axis=0, how='any')
print(f"Shape after post-processing: {enhanced_df.shape}")


# Save the enhanced DataFrame
output_path = 'stock_data/processed/merged_stock_data_with_enhanced_features___.parquet'
print(f"Saving enhanced DataFrame to {output_path}...")
enhanced_df.to_parquet(
    output_path,
    engine='pyarrow',
    index=True,
    compression='snappy'
)
print("DataFrame saved successfully.")

DataFrame structure: 2 levels
Column levels: ['Industry', 'Symbol']
Starting initial feature calculation for all stocks...


/var/folders/gn/mtbdxhjn1697bp_qhyd6y0cr0000gn/T/ipykernel_18221/622331217.py:39: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return_1d = price.pct_change(1) # Calculate once
/var/folders/gn/mtbdxhjn1697bp_qhyd6y0cr0000gn/T/ipykernel_18221/622331217.py:41: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  new_columns[(industry, symbol, 'return_3d')] = price.pct_change(3)
/var/folders/gn/mtbdxhjn1697bp_qhyd6y0cr0000gn/T/ipykernel_18221/622331217.py:42: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-lead

Constructing final DataFrame from calculated features...
Initial result_df shape: (796, 11570)
Finished calculating features.
Starting post-processing (fillna, dropna)...
Shape before dropping rows with NaN target: (796, 11570)
Shape after post-processing: (796, 11570)
Saving enhanced DataFrame to stock_data/processed/merged_stock_data_with_enhanced_features___.parquet...
DataFrame saved successfully.


In [138]:
enhanced_df[100:105]

Industry                  Automobile and Auto Components            \
Symbol                                        APOLLOTYRE             
Feature                                           BB_pos  BB_width   
2022-06-07 00:00:00+05:30                       0.420697  0.140569   
2022-06-08 00:00:00+05:30                       0.416351  0.116981   
2022-06-09 00:00:00+05:30                       0.179449  0.089678   
2022-06-10 00:00:00+05:30                      -0.237233  0.090403   
2022-06-13 00:00:00+05:30                      -0.314457  0.138326   

Industry                                                                 \
Symbol                                                                    
Feature                        EMA_10      EMA_20       EMA_5 MACD_hist   
2022-06-07 00:00:00+05:30  209.383414  206.766122  209.171555 -0.672632   
2022-06-08 00:00:00+05:30  208.711951  206.663670  208.011160 -0.978879   
2022-06-09 00:00:00+05:30  207.553969  206.252182  206.121789 -1.372455   
2022-06-10 00:00:00+05:30  205.168812  205.126794  202.226395 -2.088024   
2022-06-13 00:00:00+05:30  201.312130  203.110629  196.136618 -3.127223   

Industry                                                                  ...  \
Symbol                                                                    ...   
Feature                       RSI_14      SMA_10      SMA_20       SMA_5  ...   
2022-06-07 00:00:00+05:30  49.706704  210.216528  206.929848  210.900540  ...   
2022-06-08 00:00:00+05:30  51.057972  210.138908  207.723017  209.377267  ...   
2022-06-09 00:00:00+05:30  46.810056  209.658640  208.331841  206.786734  ...   
2022-06-10 00:00:00+05:30  38.633045  208.227539  208.319714  203.313281  ...   
2022-06-13 00:00:00+05:30  30.923995  205.278017  207.313092  198.209836  ...   

Industry                         Textiles                                     \
Symbol                         WELSPUNLIV                                      
Feature                   dist_from_sma20 ema10_ema20_cross ema5_ema10_cross   
2022-06-07 00:00:00+05:30        0.034229                 0                1   
2022-06-08 00:00:00+05:30        0.059792                 0                1   
2022-06-09 00:00:00+05:30        0.076895                 0                1   
2022-06-10 00:00:00+05:30        0.092237                 0                1   
2022-06-13 00:00:00+05:30        0.008246                 0                1   

Industry                                                            \
Symbol                                                               
Feature                        price return_1d return_3d return_5d   
2022-06-07 00:00:00+05:30  69.976631 -0.029696  0.000712  0.074981   
2022-06-08 00:00:00+05:30  72.168076  0.031317  0.004158  0.079732   
2022-06-09 00:00:00+05:30  73.761856  0.022084  0.022790  0.054843   
2022-06-10 00:00:00+05:30  75.256012  0.020256  0.075445  0.047124   
2022-06-13 00:00:00+05:30  69.727600 -0.073461 -0.033817 -0.033149   

Industry                                                              
Symbol                                                                
Feature                   return_ratio volatility_10d volatility_20d  
2022-06-07 00:00:00+05:30    -0.029696       0.030326       0.060777  
2022-06-08 00:00:00+05:30     0.031317       0.027356       0.045471  
2022-06-09 00:00:00+05:30     0.022084       0.027139       0.045085  
2022-06-10 00:00:00+05:30     0.020256       0.025079       0.045082  
2022-06-13 00:00:00+05:30    -0.073461       0.037010       0.047568  

[5 rows x 11570 columns]